# Protocol for Training Interpretable GNN Models Using HCat-GNet in Chemical Databases

This notebook contains the code necesary to replicate the experiments and obtain the results reported in the paper [HCat-GNet: a Human-Interpretable GNN Tool for Ligand Optimization in Asymmetric Catalysis](https://www.cell.com/iscience/fulltext/S2589-0042(25)00141-5). We aim to provide a simple list of steps to replicate all the results in a few lines of code, and potentially apply this protocol to *any* chemical database of interest.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EdAguilarB/hcatgnet/blob/core/colab_notebooks/paper_experiments.ipynb)

## Prepare your environment

First, make sure your computer is in good shape to run the experiments.

In [ ]:
from pathlib import Path

In [ ]:

# Get the current directory name
current_dir = Path.cwd().name

if current_dir == "colab_notebooks":
    %cd ..
    print("✅ Moved to the previous directory.")
else:
    repo_url = "https://github.com/EdAguilarB/hcatgnet.git"
    repo_name = repo_url.split("/")[-1].replace(".git", "")  # Extract repo name

    if Path(repo_name).exists():
        print(f"⚠️ Repository '{repo_name}' already exists. Skipping clone.")
    else:
        !git clone {repo_url}
        print(f"✅ Successfully cloned '{repo_name}'.")
    %cd {repo_name}
    print("✅ Moved to the repository directory.")
    %pip install -r requirements.txt

## Training on the RhCAA dataset

First, let's import all the necesary functions that are going to allow us to perform the experiments.

In [ ]:
from scripts_experiments.train_GNN import train_network_nested_cv
import pandas as pd
from data.rhcaa import rhcaa_diene

In the cell of code bellow, we define variables that are necessary for our code to work:
- `mol_cols` is a list containing the name of the columns in the csv containing the smiles of the molecules that interact to give an experimental observable (for our case, enantioselectivity). For our database, this will be: 'Ligand', 'substrate', and 'boron reagent'.
- `folds` is an integer containing the information of how many folds we wish to split dataset into. In our paper, this is set to 10.
- `global_seed` is an integer defined to allow reproducibility of experiments, so that each time the training is run, it delivers the same results. We defined this as 20232023 in our experiments.
- `include_Hs` is whether or not to include explicit hydrogens in the molecular graph representation.
- `root` is the path where our dataset is stored. Our software relies on `torch_geometric`, and as so, the last directory where our dataset is stored must be called 'raw', however, the raw must be ommited from the path definiction, therefore, if the file is in the directory "data/datasets/rhcaa/known_unknown/learning/raw", we only specify "data/datasets/rhcaa/known_unknown/learning", as done below.
- `filename` is the name of the file we wish to model.

In [ ]:
filename = "rhcaa.csv"
root = Path("data") / "datasets" / "rhcaa" / "known_unknown" / "learning"
mol_cols = ["Ligand", "substrate", "boron reagent"]
target_variable = "ddG"
include_Hs = True
folds = 10
global_seed = 20232023

### Training the GNN model

`dataset` is a variable that is going to store our molecular graph representation dataset. This is built using the class `rhcaa_diene`, which expects some of the arguments we defined below. This dataset is going to be the one used by our model to learn correlations between molecular graph structure and observale property (enantioselectivity).

In [ ]:
dataset = rhcaa_diene(
    filename=filename,
    root=root,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=folds,
    random_seed=global_seed,
)

`dir_results` is the parent directory where we wish to save the results obtained from the model training and evaluation.

In [ ]:
dir_results = Path("results") / dataset._name / "learning"

`dir_results_GNN` is where the GNN results will be logged.

In [ ]:
dir_results_GNN = dir_results / "results_GNN"

By running the `train_network_nested_cv` function, the model is trained, using the dataset provided as argument of the function.

In [ ]:
train_network_nested_cv(
    graph_dataset=dataset,
    log_results_dir=dir_results_GNN,
    folds=folds,
    global_seed=global_seed,
)

### Training the traditional machine learning models. 

After training the GNN models, we train the TML models to compare the performance of both approaches. First, import the `train_tml_model_nested_cv` function.

In [ ]:
from scripts_experiments.train_TML import train_tml_model_nested_cv

Then, we read the csv that contains the information of descriptors of the molecules (steric and electronic parameters), the target variable to model (ddG), and the folds to split the data (this is to make sure that both GNN and TML are trained and evaluated using the exact same sets of data). To read the data, we use the `pd.read_csv` function, and provide the path where we can find such csv. Notice that the path is the same as the one used for the molecular graph dataset (`root`), but in this case we must include the 'raw' directory and the file name, so that the function nows that is the file to be read.

In [ ]:
data = pd.read_csv(root / "raw" / dataset.filename)
data.sample(10, random_state=global_seed)

Before using the csv file as input, it requires some preprocessing. First, it is necessay to define the name of the columns containing the descriptors of the molecules. To do this, we store the names as list called `descriptors`. After that, we eliminate all those columns that are not necessaty for the model training. We keep only the descriptors, the target variable ddG, the fold each point corresponds to, and the index of the reaction for identification porpuses.

In [ ]:
descriptors = [
    "LVR1",
    "LVR2",
    "LVR3",
    "LVR4",
    "LVR5",
    "LVR6",
    "LVR7",
    "VB",
    "ER1",
    "ER2",
    "ER3",
    "ER4",
    "ER5",
    "ER6",
    "ER7",
    "SStoutR1",
    "SStoutR2",
    "SStoutR3",
    "SStoutR4",
    "temp",
]

data = data[descriptors + [target_variable, "fold", "index"]]
data.sample(10, random_state=global_seed)

We then define the directory to save the TML results.

In [ ]:
def dir_results_TML(algorithm: str, representation: str) -> Path:
    return dir_results / "results_TML" / algorithm / representation

The function `train_tml_model_nested_cv` allows to train the TML model using the input data provided. Similar to the GNN training function, it expects from us to tell the name of the columns where the molecule's smiles are stored (mol_cols), the directory where the results will be saved, and the global seed. Also, it expects as an input which TML algorithm to use for training. For random forest, input 'rf', for Gradient Boosting 'gb', and for linear regression 'lr'.

In [ ]:
train_tml_model_nested_cv(
    data_csv=data,
    mol_cols=mol_cols,
    descriptors=descriptors,
    target_variable=target_variable,
    log_results_dir=dir_results_TML("rf", "bespoke"),
    tml_algorithm="rf",
    global_seed=global_seed,
)

train_tml_model_nested_cv(
    data_csv=data,
    mol_cols=mol_cols,
    descriptors=descriptors,
    target_variable=target_variable,
    log_results_dir=dir_results_TML("gb", "bespoke"),
    tml_algorithm="gb",
    global_seed=global_seed,
)

train_tml_model_nested_cv(
    data_csv=data,
    mol_cols=mol_cols,
    descriptors=descriptors,
    target_variable=target_variable,
    log_results_dir=dir_results_TML("lr", "bespoke"),
    tml_algorithm="lr",
    global_seed=global_seed,
)

### Plot and compare results

We created a function that allows direct comparison of results between the two methodologies, including comparison of metrics per fold, and statistical tests to  determine if there was or not signifficant difference. We import the `plot_results` function and pass the parameters of:
- save_path: path where we want the results to be stored
- experiments_gnn_path: path where we stored the GNN experiments.
- experiments_tml_path: path where we stored the TML experiments.

In [ ]:
from scripts_experiments.compare_gnn_tml import plot_results

Generate a path to save the comparison results.

In [ ]:
def comparison_path_generator(log_dir, tml_algorithm, descriptors):
    return log_dir / "comparison" / f"GNN_vs_{tml_algorithm}" / descriptors

Generate the comparisons.

In [ ]:
plot_results(
    save_dir=comparison_path_generator(dir_results, "rf", "bespoke"),
    experiments_gnn_path=dir_results_GNN,
    experiments_tml_path=dir_results_TML("rf", "bespoke"),
    tml_algorithm="rf",
)

plot_results(
    save_dir=comparison_path_generator(dir_results, "gb", "bespoke"),
    experiments_gnn_path=dir_results_GNN,
    experiments_tml_path=dir_results_TML("gb", "bespoke"),
    tml_algorithm="gb",
)

plot_results(
    save_dir=comparison_path_generator(dir_results, "lr", "bespoke"),
    experiments_gnn_path=dir_results_GNN,
    experiments_tml_path=dir_results_TML("lr", "bespoke"),
    tml_algorithm="lr",
)

At this point, you should be able to see the results in the `dir_results` path you provided for the 'learning' set.

Lastly, we can visualise the results of each TML algorithm compared to the GNN model in a parity plot using the code below.

In [ ]:
from hcatgnet.utils.plot_utils import plot_mean_predictions

In [ ]:
results_rf = pd.read_csv(
    comparison_path_generator(dir_results, "rf", "bespoke") / "predictions_all.csv"
)
results_gb = pd.read_csv(
    comparison_path_generator(dir_results, "gb", "bespoke") / "predictions_all.csv"
)
results_lr = pd.read_csv(
    comparison_path_generator(dir_results, "lr", "bespoke") / "predictions_all.csv"
)

In [ ]:
plot_mean_predictions(results_rf)

In [ ]:
plot_mean_predictions(results_gb)

In [ ]:
plot_mean_predictions(results_lr)

### Evaluation of the models on the Unseen Set

Import the necessary libraries and functions.

In [ ]:
from scripts_experiments.predict_test import predict_final_test

Similarly as before, we need to provide information about where in our computer the unseen set is stored. Same as the case before, we need to provide the root (where it is stored excluding the 'raw' directory), the filename, and the path where we would like to store the results.

In [ ]:
root_test = Path("data") / "datasets" / "rhcaa" / "known_unknown" / "test"
filename_test = "rhcaa.csv"
dir_results_test = Path("results") / dataset._name / "test"

Now let's create the molecular graph dataset.

In [ ]:
test_graph_dataset = rhcaa_diene(
    filename=filename_test,
    root=root_test,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=None,
)

Read the csv containing the unseen set.

In [ ]:
test_dataset = pd.read_csv(root_test / "raw" / filename_test)
test_dataset = test_dataset[descriptors + [target_variable, "index"]]
test_dataset.sample(10, random_state=global_seed)

We then define the paths to log the results.

In [ ]:
dir_results_GNN_test = dir_results_test / "results_GNN"


def dir_results_TML_test(algorithm: str, representation: str) -> Path:
    return dir_results_test / "results_TML" / algorithm / representation

Now use the `predict_final_test` function to get the predictions.

In [ ]:
predict_final_test(
    graph_dataset=test_graph_dataset,
    handcrafted_descriptors_dataset=test_dataset,
    descriptors=descriptors,
    GNN_experiment_path=dir_results_GNN,
    TML_experiment_path=dir_results_TML("gb", "bespoke"),
    GNN_log_dir=dir_results_GNN_test,
    TML_log_dir=dir_results_TML_test("gb", "bespoke"),
)

Lastly, we use again the `plot_results` function to get comparisons between the two approaches for this dataset.

In [ ]:
plot_results(
    save_dir=comparison_path_generator(dir_results_test, "gb", "bespoke"),
    experiments_gnn_path=dir_results_GNN_test,
    experiments_tml_path=dir_results_TML_test("gb", "bespoke"),
    tml_algorithm="gb",
)

Lastly, we can plot the results obtained for the unseen set as shown below.

In [ ]:
results = pd.read_csv(
    comparison_path_generator(dir_results_test, "gb", "bespoke") / "predictions_all.csv"
)
results.sample(10, random_state=global_seed)

In [ ]:
plot_mean_predictions(results)

## Training HCat-GNet using part of the "unseen" ligand data

To perform this experiments, we randomly took some samples from the 'unseen' ligand data and included them into the 'seen' or learning data. By performing this experiment, we aimed to evaluate how the predictions of the model on the new ligand family changed when seeing some of this samples. A dataset with half of the 'unseen' samples included in the 'seen' data is available in `data/datasets/rhcaa/halftest/learning/raw/rhcaa_half.csv`, while the half of data points excluded is available in `data/datasets/rhcaa/halftest/test/raw/rhcaa_half_test.csv`. Lastly, a dataset containing all the combined data is available in `data/datasets/rhcaa/all_combined/raw/rhcaa_all.csv`. We will use this files to replicate the Figure 8 in the paper.

### Including half of the unseen set into the learning data

We define the variables that are useful to find our data.

In [ ]:
root = Path("data") / "datasets" / "rhcaa" / "halftest" / "learning"
filename = "rhcaa_half.csv"

We create a molecular graph dataset from which the GNN will learn correlations.

In [ ]:
dataset = rhcaa_diene(
    filename=filename,
    root=root,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=folds,
    random_seed=global_seed,
)

We define the path where we wish to store the results.

In [ ]:
dir_results = Path("results") / f"{dataset._name}_half" / "learning"
dir_results_GNN = dir_results / "results_GNN"

Then, we run the GNN training as done before.

In [ ]:
train_network_nested_cv(
    graph_dataset=dataset,
    log_results_dir=dir_results_GNN,
    folds=folds,
    global_seed=global_seed,
)

Then, the csv with the data is loaded to train the TML methods.

In [ ]:
data = pd.read_csv(root / "raw" / dataset.filename)
data.sample(10, random_state=global_seed)

We filter the columns of the csv to retain only those useful for training.

In [ ]:
data = data[descriptors + [target_variable, "fold", "index"]]
data.sample(10, random_state=global_seed)

Then, we run the TML model training

In [ ]:
train_tml_model_nested_cv(
    data_csv=data,
    mol_cols=mol_cols,
    descriptors=descriptors,
    target_variable=target_variable,
    log_results_dir=dir_results_TML("gb", "bespoke"),
    tml_algorithm="gb",
    global_seed=global_seed,
)

To compare the results, we use the `plot_results` function again.

In [ ]:
plot_results(
    save_dir=comparison_path_generator(dir_results, "gb", "bespoke"),
    experiments_gnn_path=dir_results_GNN,
    experiments_tml_path=dir_results_TML("gb", "bespoke"),
    tml_algorithm="gb",
)

Now, we load the other half test data to make predictions

In [ ]:
root_test = Path("data") / "datasets" / "rhcaa" / "halftest" / "test"
filename_test = "rhcaa_half_test.csv"
dir_results_test = Path("results") / f"{dataset._name}_half" / "test"

We create the molecular graph dataset.

In [ ]:
test_graph_dataset = rhcaa_diene(
    filename=filename_test,
    root=root_test,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=None,
)

Read the csv containing the unseen set.

In [ ]:
test_dataset = pd.read_csv(root_test / "raw" / filename_test)
test_dataset = test_dataset[descriptors + [target_variable, "index"]]
test_dataset.sample(10, random_state=global_seed)

Create the paths to store results.

In [ ]:
dir_results_GNN_test = dir_results_test / "results_GNN"

Make predictions on the unseen set.

In [ ]:
predict_final_test(
    graph_dataset=test_graph_dataset,
    handcrafted_descriptors_dataset=test_dataset,
    descriptors=descriptors,
    GNN_experiment_path=dir_results_GNN,
    TML_experiment_path=dir_results_TML("gb", "bespoke"),
    GNN_log_dir=dir_results_GNN_test,
    TML_log_dir=dir_results_TML_test("gb", "bespoke"),
)

We then compare the results of approaches.

In [ ]:
plot_results(
    save_dir=comparison_path_generator(dir_results_test, "gb", "bespoke"),
    experiments_gnn_path=dir_results_GNN_test,
    experiments_tml_path=dir_results_TML_test("gb", "bespoke"),
    tml_algorithm="gb",
)

Now, we can analyse the results and get a parity plot.

In [ ]:
results_seen = pd.read_csv(
    comparison_path_generator(dir_results, "gb", "bespoke") / "predictions_all.csv"
)
plot_mean_predictions(results_seen)

For the unseen set, it can be done as shown.

In [ ]:
results_unseen = pd.read_csv(
    comparison_path_generator(dir_results_test, "gb", "bespoke") / "predictions_all.csv"
)
plot_mean_predictions(results_unseen)

However, in the paper, we present the parity plot of all the reactions that belong to the original unseen set. To do that, we can use the following code.

In [ ]:
results_unseen_all = pd.concat(
    [results_seen.loc[results_seen["index"] > 667], results_unseen]
)
plot_mean_predictions(results_unseen_all)

### Including all the unseen set into the learning data

In [ ]:
root = Path("data") / "datasets" / "rhcaa" / "all_combined" / "learning"
filename = "rhcaa_all.csv"

In [ ]:
dataset = rhcaa_diene(
    filename=filename,
    root=root,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=folds,
    random_seed=global_seed,
)

In [ ]:
dir_results = Path("results") / f"{dataset._name}_all"

In [ ]:
dir_results_GNN = dir_results / "results_GNN"

In [ ]:
train_network_nested_cv(
    graph_dataset=dataset,
    log_results_dir=dir_results_GNN,
    folds=folds,
    global_seed=global_seed,
)

Now, load the dataset with descriptors to train TML models.

In [ ]:
data = pd.read_csv(root / "raw" / dataset.filename)
data.sample(10, random_state=global_seed)

Clean the database.

In [ ]:
data = data[descriptors + [target_variable, "fold", "index"]]
data.sample(10, random_state=global_seed)

In [ ]:
train_tml_model_nested_cv(
    data_csv=data,
    mol_cols=mol_cols,
    descriptors=descriptors,
    target_variable=target_variable,
    log_results_dir=dir_results_TML("gb", "bespoke"),
    tml_algorithm="gb",
    global_seed=global_seed,
)

We now compare results.

In [ ]:
plot_results(
    save_dir=comparison_path_generator(dir_results, "gb", "bespoke"),
    experiments_gnn_path=dir_results_GNN,
    experiments_tml_path=dir_results_TML("gb", "bespoke"),
    tml_algorithm="gb",
)

In [ ]:
results_all = pd.read_csv(
    comparison_path_generator(dir_results, "gb", "bespoke") / "predictions_all.csv"
)
plot_mean_predictions(results_all)

And now we filter to only get the reactions that belong to the original unseen set.

In [ ]:
plot_mean_predictions(results_all.loc[results_all["index"] > 667])

## Runing Interpretations

As shown in the main paper, our tool allows to interpret the GNN models trained. We are going to demonstrate how to get such intepretations using code. First, import the necessary functions.

In [ ]:
from hcatgnet.gnn_interpretation.mol_denoiser import (
    denoise_mol,
    explain_node_feats,
    shapley_analysis,
)
import torch

For the paper, we randomly selected one of the models to make all the explanation plots shown using a random number generator. Those numbers led to analyse the model correspoding to the outer fold 8, inner fold 10 model. We create the path to load the model, loaders, and other data from such run.

In [ ]:
test_fold = 8
val_fold = 10

model_path = (
    Path("TRIAL")
    / "rhcaa_diene"
    / "learning"
    / "results_GNN"
    / f"Fold_{test_fold}_test_set"
    / f"Fold_{val_fold}_val_set"
)

Now, we load the previously trained model using the code bellow.

In [ ]:
model = torch.load(model_path / "model.pth", weights_only=False)
params = torch.load(model_path / "model_params.pth", weights_only=True)
model.load_state_dict(params)

To replicate Figure 9 in the main paper, we ran a GNNExplainer algorithm using the model we loaded before on the test set of such run. To replicate it, we first load our molecular graph test dataset.

In [ ]:
test_dataset = torch.load(model_path / "test_loader.pth").dataset

Lastly, we use the `explain_node_feats` function, which expects the model and the data for which we want to run the analysis.

In [ ]:
explain_node_feats(
    model=model,
    dataset=test_dataset,
    include_Hs=include_Hs,
    feature_sizes=dataset.atom_feats_length,
)

To recreate figure 10, we make use of the function `denoise_mol`. This function relies on GNNExplainer to assign a score to each node feature within each node. Then, we assign such score a value of transparency an it is mapped to the molecule. The result is a plot of the molecule with atoms with transparecy values based on how important is a certain property in that specific node. The arguments that this function expect are:
- `model` is the model to be explained.
- `mol_dataset` is the molecular graph dataset to use to get the explanations from.
- `mol_index` is the index of the molecule/reaction within the dataset to plot.
- `denoise_mol` is which one of the molecules used in the molecular graph representation to plot.
- `include_Hs` is whether or not explicit hydrogens were used to build the molecular graph representations.
- `analyse_feature` is a parameter to allow get the importance from a specific type of node feature. For the case of our dataset, the allowed options are:
    - "Atomic Identity"
    - "Atom Degree"
    - "Atom Hybridization"
    - "Atom Aromaticity"
    - "Atom in Ring"
    - "Atom Chirality"
    - "Ligand Confg."
- `norm_denoise` is whether or not to normalise the scores between 0 and 1.


In [ ]:
for feat in dataset.atom_feats_length.keys():

    if feat:
        print("Explaining feature:", feat)

    denoise_mol(
        model=model,
        mol_dataset=dataset,
        mol_index=1,
        denoise_mol="Ligand",
        analyse_feature=feat,
        include_Hs=include_Hs,
    )

Lastly, with the `shapley_analysis` function we can compute scores using Shapley Value Sampling. This function expects the model to be explained, the dataset where the molecule that wants to be exlained is, the index of the molecule to explain, and the name of the molecule to plot.

In [ ]:
explain_mols = [82, 416, 89, 91, 95, 97]

for mol in explain_mols:
    shapley_analysis(model=model, dataset=dataset, explain_mol=mol, plot_mol="ligand")

## Evaluation of HCat-GNet on Other Asymmetric Reactions

### The RhCAA BiAryl Dataset

The `rhcaa_biaryl` class will be useful to create the graph representation for this dataset.

In [ ]:
from data.biaryl import rhcaa_biaryl

Then, we define the variables that allow the creation of the graph dataset.

In [ ]:
filename = "biaryl.csv"
root = Path("data") / "datasets" / "biAryl" / "learning"
mol_cols = ["Ligand", "substrate", "boron reagent"]

The `rhcaa_biaryl` dataset expects the same inputs of `rhcaa_diene` that we explained before.

In [ ]:
dataset = rhcaa_biaryl(
    filename=filename,
    root=root,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=folds,
    random_seed=global_seed,
)

Define the directories to save the results.

In [ ]:
dir_results = Path("results") / dataset._name / "learning"
dir_results_GNN = dir_results / "results_GNN"

Train the GNN model using the `train_network_nested_cv` function.

In [ ]:
train_network_nested_cv(
    graph_dataset=dataset,
    log_results_dir=dir_results_GNN,
    folds=folds,
    global_seed=global_seed,
)

For this example, we won't use traditional approaches to model the dataset. Therefore, we use the `plot_results_GNN` function from `scripts_experiments.summarize_gnn_results` to get the parity plot and a residulas distribution plot.

In [ ]:
from scripts_experiments.summarize_gnn_results import plot_results_GNN

The `plot_results_GNN` expects only three arguments:
- `experiments_gnn` is the directory that where the results of the GNN model training are logged.
- `folds` is the number of folds used for training.
- `log_dir_results` is the directory where the results (plots) will be saved.

In [ ]:
plot_results_GNN(
    experiments_gnn=dir_results_GNN, folds=folds, log_dir_results=dir_results
)

Now, we define the variables for the test set.

In [ ]:
filename_test = "biaryl.csv"
root_test = Path("data") / "datasets" / "biAryl" / "test"
dir_results_test = Path("results") / dataset._name / "test"

Create a molecular graph dataset for the unseen set.

In [ ]:
test_graph_dataset = rhcaa_biaryl(
    filename=filename_test,
    root=root_test,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=None,
)

We import the function `predict_final_test_GNN` from `scripts_experiments.predict_test_GNN` to get the predictions.

In [ ]:
from scripts_experiments.predict_test_GNN import predict_final_test_GNN

Define the path where the results will be saved.

In [ ]:
dir_results_test = Path("results") / dataset._name / "test"
dir_results_GNN_test = dir_results_test / "results_GNN"

Run the `predict_final_test_GNN` function to get the predictions.

In [ ]:
predict_final_test_GNN(
    graph_dataset=test_graph_dataset,
    folds=folds,
    GNN_experiment_path=dir_results_GNN,
    log_results_dir=dir_results_GNN_test,
)

Get plots of the results.

In [ ]:
plot_results_GNN(dir_results_GNN_test, folds=folds, log_dir_results=dir_results_test)

### Hypervalent Iodine (III) dataset

For this dataset, all the code will remain the same, only the way of how the molecular graph dataset is created changes. First, we import the `hypervalent_iodine` class.

In [ ]:
from data.hypervalent_iodine import hypervalent_iodine

Then, we define the variables related to this dataset. The most important difference is that in this case mol_cols had to be redefined to match the information in the csv.

In [ ]:
filename = "hypervalent_iodine.csv"
root = Path("data") / "datasets" / "hypervalent_iodine" / "learning"
mol_cols = ["Sub", "Precat", "additive1", "additive2", "Solvent1", "Solvent2"]

Create the dataset using the `hypervalent_iodine` class. This class accepts the same arguments as the `rhcaa` classes.

In [ ]:
dataset = hypervalent_iodine(
    filename=filename,
    root=root,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=folds,
    random_seed=global_seed,
)

Define the directories where the results will be saved.

In [ ]:
dir_results = Path("results") / dataset._name / "learning"
dir_results_GNN = dir_results / "results_GNN"

Run the GNN model training.

In [ ]:
train_network_nested_cv(
    graph_dataset=dataset,
    log_results_dir=dir_results_GNN,
    folds=folds,
    global_seed=global_seed,
)

Get the plots of the model.

In [ ]:
plot_results_GNN(
    experiments_gnn=dir_results_GNN, folds=folds, log_dir_results=dir_results
)

Define the variables for the test (unseen) set.

In [ ]:
filename_test = "hypervalent_iodine.csv"
root_test = Path("data") / "datasets" / "hypervalent_iodine" / "test"
dir_results_test = Path("results") / dataset._name / "test"

Create the molecular graph dataset.

In [ ]:
test_graph_dataset = hypervalent_iodine(
    filename=filename_test,
    root=root_test,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=None,
)

Define the directories where the results will be logged.

In [ ]:
dir_results_test = Path("results") / dataset._name / "test"
dir_results_GNN_test = dir_results_test / "results_GNN"

Get predictions of the unseen set.

In [ ]:
predict_final_test_GNN(
    graph_dataset=test_graph_dataset,
    folds=folds,
    GNN_experiment_path=dir_results_GNN,
    log_results_dir=dir_results_GNN_test,
)

Plot the results.

In [ ]:
plot_results_GNN(dir_results_GNN_test, folds=folds, log_dir_results=dir_results_test)

### N,S-Acetal Formation Dataset

For this dataset, we use the `general_reaction.reaction_representation` class to create the dataset.

In [ ]:
from data.general_reaction import reaction_representation

Define the name of your file, where's located and the columns with smiles.

In [ ]:
filename = "N_S_acetal.csv"
root = Path("data") / "datasets" / "N_S_acetal" / "learning"
mol_cols = ["Catalyst", "Imine", "Thiol"]

Create the molecular graph dataset.

In [ ]:
dataset = reaction_representation(
    filename=filename,
    root=root,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=folds,
    random_seed=global_seed,
)

Define the directories to save the results.

In [ ]:
dir_results = Path("results") / dataset._name / "learning"
dir_results_GNN = dir_results / "results_GNN"

Run the GNN model training.

In [ ]:
train_network_nested_cv(
    graph_dataset=dataset,
    log_results_dir=dir_results_GNN,
    folds=folds,
    global_seed=global_seed,
)

Plot the results.

In [ ]:
plot_results_GNN(
    experiments_gnn=dir_results_GNN, folds=folds, log_dir_results=dir_results
)

Define variables of your unseen data.

In [ ]:
filename_test = "N_S_acetal.csv"
root_test = Path("data") / "datasets" / "N_S_acetal" / "test"
dir_results_test = Path("results") / dataset._name / "test"

In [ ]:
test_graph_dataset = reaction_representation(
    filename=filename_test,
    root=root_test,
    molcols=mol_cols,
    target_variable=target_variable,
    include_Hs=include_Hs,
    num_folds=None,
)

Define the directories to log the results.

In [ ]:
dir_results_test = Path("results") / dataset._name / "test"
dir_results_GNN_test = dir_results_test / "results_GNN"

Predict the target variable for your unseen set.

In [ ]:
predict_final_test_GNN(
    graph_dataset=test_graph_dataset,
    folds=folds,
    GNN_experiment_path=dir_results_GNN,
    log_results_dir=dir_results_GNN_test,
)

Plot the results.

In [ ]:
plot_results_GNN(dir_results_GNN_test, folds=folds, log_dir_results=dir_results_test)